# Fine-tune Whisper-small for Kenyan Swahili with JengaNLP (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rogendo/JengaNLP/blob/main/notebooks/whisper_small_swahili_gpu.ipynb)

Trains `openai/whisper-small` on Google FLEURS Kenyan Swahili (`sw_ke`, ~10 h of read speech, CC-BY-4.0)
with `jenga-nlp asr train`, then compares word error rate before and after on the held-out test split.

**Needs a GPU.** In Colab: *Runtime → Change runtime type → T4 GPU*. Also works on any Jupyter machine with
a CUDA GPU (A10, A40, L4, ...). Rough time on a T4: 1–1.5 h for 2,000 steps plus evaluation.

Disk: ~3.7 GB download, ~1.5 GB of audio clips, ~3 GB per checkpoint (2 kept).

## 1. Check the GPU and install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Whisper fixes are on GitHub main (not yet in the PyPI release)
%pip install -q "jenga-nlp[asr] @ git+https://github.com/Rogendo/JengaNLP.git@main" pandas pyarrow
!git clone -q --depth 1 https://github.com/Rogendo/JengaNLP.git jenganlp_src 2>/dev/null || git -C jenganlp_src pull -q

In [ ]:
import torch, jenga_nlp
print(jenga_nlp.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch the runtime')

## 2. Download FLEURS Kenyan Swahili and write manifests
One JSONL row per clip: `{"audio": path, "text": transcript, "duration_s": seconds}`. Train sentences that also
occur in validation or test are removed, so the test score is on unseen sentences.

In [ ]:
!python jenganlp_src/examples/prepare_fleurs_asr.py --lang sw_ke --out data/fleurs_sw_ke
!rm -rf data/fleurs_sw_ke/raw   # parquet files are no longer needed

## 3. Baseline: how good is whisper-small before training?
Scored on the test split (487 clips), decoding forced to Swahili.

In [ ]:
!python jenganlp_src/examples/compare_asr_wer.py --manifest data/fleurs_sw_ke/test.jsonl --language sw \
    --models openai/whisper-small --out baseline.json

## 4. Fine-tune
The config is `examples/configs/whisper_small_sw_gpu.yaml`: fp16, gradient checkpointing, batch 16,
2,000 steps, WER on the validation split every 250 steps, best checkpoint kept.

- Out of memory: add `batch_size: 8` and `gradient_accumulation_steps: 2` to the config.
- Colab disconnected: re-run steps 1 and 4 with `--resume latest` (keep the `results/` folder, e.g. on Google Drive).
- The live log is in `train.log`; the next cell prints new WER lines.

In [ ]:
!jenga-nlp asr train --config jenganlp_src/examples/configs/whisper_small_sw_gpu.yaml 2>&1 | tee train.log | grep --line-buffered -E "'loss'|eval_wer|Error|Traceback"

## 5. Before vs after on the test split

In [ ]:
!python jenganlp_src/examples/compare_asr_wer.py --manifest data/fleurs_sw_ke/test.jsonl --language sw \
    --models openai/whisper-small results/whisper-small-sw --out comparison.json

## 6. Transcribe your own audio
Upload a WAV/FLAC/OGG/MP3 file (any sample rate) in the Colab file panel and set its name below.

In [ ]:
from jenga_nlp.asr.config import WhisperConfig
from jenga_nlp.asr.pipeline import WhisperPipeline

asr = WhisperPipeline(WhisperConfig(model_name="results/whisper-small-sw", language="sw", device="auto"))
audio_file = "my_voice.wav"  # @param {type:"string"}
print(asr.transcribe(audio_file)["text"])

## 7. Optional: keep the model
Colab deletes files when the session ends. Save to Google Drive, or push to the Hub (private).

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r results/whisper-small-sw /content/drive/MyDrive/
# from huggingface_hub import notebook_login, upload_folder
# notebook_login()
# upload_folder(folder_path="results/whisper-small-sw", repo_id="<you>/whisper-small-sw-ke", repo_type="model")